In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_4.py ──
"""
Shared infrastructure for MLFP03 Exercise 4 — Gradient Boosting Deep Dive.

Contains: Singapore credit-scoring data loader, preprocessing pipeline
wrapper, numpy/polars conversion, model-zoo factories (XGBoost/LightGBM/
CatBoost with identical defaults), evaluation metric helpers, and the
output directory used by every technique file.

Technique-specific code (from-scratch boosting loops, split-gain formulas,
hyperparameter sweeps, early-stopping analysis) does NOT belong here — it
lives in the per-technique files under `modules/mlfp03/solutions/ex_4/`.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONFIG — output directory, random seeds, dataset tag
# ════════════════════════════════════════════════════════════════════════

SEED = 42
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"

OUTPUT_DIR = Path("outputs") / "mlfp03" / "ex_4_boosting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit-scoring, shared across all four techniques
# ════════════════════════════════════════════════════════════════════════


def load_credit_data() -> pl.DataFrame:
    """Load the Singapore credit-scoring dataset via MLFPDataLoader."""
    loader = MLFPDataLoader()
    return loader.load(DATASET_MODULE, DATASET_FILE)


def prepare_credit_split() -> dict[str, Any]:
    """Load credit data and return a train/test split ready for boosting.

    Returns a dict with:
      X_train, y_train, X_test, y_test : numpy arrays
      feature_names                    : list[str]
      default_rate                     : float (positive-class prevalence)

    Tree models do not need normalisation, so we set ``normalize=False``.
    Categoricals are ordinal-encoded because XGBoost/LightGBM expect numeric
    input; CatBoost would accept raw categoricals but we keep the pipeline
    consistent across all three libraries for a fair comparison.
    """
    credit = load_credit_data()

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=credit,
        target=TARGET_COLUMN,
        train_size=0.8,
        seed=SEED,
        normalize=False,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=TARGET_COLUMN,
    )

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": col_info["feature_columns"],
        "default_rate": float(credit[TARGET_COLUMN].mean()),
    }


# ════════════════════════════════════════════════════════════════════════
# MODEL FACTORIES — identical defaults for fair comparison
# ════════════════════════════════════════════════════════════════════════


def make_xgboost(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build an XGBoost classifier with course-standard defaults."""
    import xgboost as xgb

    return xgb.XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        eval_metric="logloss",
        random_state=SEED,
        verbosity=0,
        **extra,
    )


def make_lightgbm(
    n_estimators: int = 500,
    learning_rate: float = 0.1,
    max_depth: int = 6,
    **extra: Any,
):
    """Build a LightGBM classifier with course-standard defaults."""
    import lightgbm as lgb

    return lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=31,
        random_state=SEED,
        verbose=-1,
        **extra,
    )


def make_catboost(
    iterations: int = 500,
    learning_rate: float = 0.1,
    depth: int = 6,
    **extra: Any,
):
    """Build a CatBoost classifier with course-standard defaults."""
    import catboost as cb

    return cb.CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        random_seed=SEED,
        verbose=0,
        **extra,
    )


# ════════════════════════════════════════════════════════════════════════
# EVALUATION — shared metric helpers for imbalanced binary classification
# ════════════════════════════════════════════════════════════════════════


def evaluate_classifier(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    threshold: float = 0.5,
) -> dict[str, float]:
    """Return the full boosting-eval metric bundle.

    AUC-PR is the primary metric — with a 12% default rate, AUC-ROC rewards
    models that rank negatives correctly even when they miss every default.
    """
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "auc_roc": float(roc_auc_score(y_true, y_proba)),
        "auc_pr": float(average_precision_score(y_true, y_proba)),
        "log_loss": float(log_loss(y_true, y_proba)),
        "brier": float(brier_score_loss(y_true, y_proba)),
        "f1": float(f1_score(y_true, y_pred)),
    }


def print_metrics(
    name: str, metrics: dict[str, float], train_time: float | None = None
) -> None:
    """One-line headline: AUC-ROC | AUC-PR | Log Loss | F1 | Time."""
    time_str = f" | time={train_time:.2f}s" if train_time is not None else ""
    print(
        f"  {name}: "
        f"AUC-ROC={metrics['auc_roc']:.4f} | "
        f"AUC-PR={metrics['auc_pr']:.4f} | "
        f"log_loss={metrics['log_loss']:.4f} | "
        f"F1={metrics['f1']:.4f}"
        f"{time_str}"
    )


# ════════════════════════════════════════════════════════════════════════
# FROM-SCRATCH GRADIENT BOOSTING (used by 01_boosting_theory.py)
# ════════════════════════════════════════════════════════════════════════


def xgb_split_gain(
    g_left: float,
    h_left: float,
    g_right: float,
    h_right: float,
    lambda_reg: float = 1.0,
    gamma: float = 0.0,
) -> float:
    """XGBoost split-gain formula from 2nd-order Taylor expansion of log-loss.

    Gain = ½ [G_L²/(H_L+λ) + G_R²/(H_R+λ) - (G_L+G_R)²/(H_L+H_R+λ)] - γ
    """
    left_score = g_left**2 / (h_left + lambda_reg)
    right_score = g_right**2 / (h_right + lambda_reg)
    parent_score = (g_left + g_right) ** 2 / (h_left + h_right + lambda_reg)
    return 0.5 * (left_score + right_score - parent_score) - gamma


def make_1d_demo(n: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Generate a 1D logistic-shaped binary classification demo.

    Used by the from-scratch boosting loop to show that residual-fitting
    converges in just a handful of rounds.
    """
    rng = np.random.default_rng(SEED)
    x = rng.uniform(0, 1, n).reshape(-1, 1)
    true_proba = 1 / (1 + np.exp(-10 * (x.ravel() - 0.5)))
    y = rng.binomial(1, true_proba)
    return x, y




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 4.3: LightGBM and CatBoost — Same Task, Faster Trees
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Train LightGBM and CatBoost on the same Singapore credit data
#   - Explain each library's split-finding strategy
#   - Compare AUC-PR, log loss, and train time across three libraries
#   - Decide which library to use based on data shape
#
# PREREQUISITES: Exercise 4.2 (XGBoost baseline).
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — exact vs histogram vs ordered boosting
#   2. Build — three models with matched hyperparameters
#   3. Train — fit each, record AUC-PR and train time
#   4. Visualise — side-by-side comparison chart
#   5. Apply — FairPrice fraud detection library choice
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

import plotly.graph_objects as go
from dotenv import load_dotenv


load_dotenv()



## THEORY — Three Libraries, One Gradient-Boosting Recipe


In [ ]:
# All three build additive trees on pseudo-residuals. They differ in
# HOW each tree finds the best split:
#
#   XGBoost:  exact (sort every feature, try every threshold)
#   LightGBM: histogram bins + GOSS + leaf-wise growth (faster on big data)
#   CatBoost: ordered boosting (no target leakage on categoricals)
#
# Practical decision tree:
#   > 1M rows, mostly numeric       → LightGBM
#   > High-cardinality categoricals → CatBoost
#   > Medium mixed data             → XGBoost



## TASK 2 — BUILD all three models with matched defaults


In [ ]:
print("\n" + "=" * 70)
print("  LightGBM + CatBoost vs XGBoost — Singapore Credit Scoring")
print("=" * 70)

data = prepare_credit_split()
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]

print(f"\n  Train: {X_train.shape} | Test: {X_test.shape}")
print(f"  Default rate: {data['default_rate']:.2%}")

# TODO: Build a dict of three models using the course-standard factories
# make_xgboost, make_lightgbm, make_catboost.
# Use n_estimators=500 (iterations=500 for CatBoost), learning_rate=0.1,
# max_depth=6 (depth=6 for CatBoost).
models = {
    "XGBoost": ____,
    "LightGBM": ____,
    "CatBoost": ____,
}



## TASK 3 — TRAIN each library and record time + metrics


In [ ]:
results: dict[str, dict] = {}
print("\n  --- Training ---")
for name, model in models.items():
    t0 = time.perf_counter()
    # TODO: Fit the model. CatBoost takes eval_set=(X_test, y_test);
    # XGBoost and LightGBM take eval_set=[(X_test, y_test)], verbose=False.
    if name == "CatBoost":
        ____
    else:
        ____
    train_time = time.perf_counter() - t0

    # TODO: Get class-1 probabilities and evaluate.
    y_proba = ____
    metrics = ____
    results[name] = {"metrics": metrics, "train_time": train_time, "model": model}
    print_metrics(name, metrics, train_time=train_time)



### Checkpoint 1


In [ ]:
for name, r in results.items():
    assert r["metrics"]["auc_roc"] > 0.7, f"{name} AUC-ROC should exceed 0.7"
    assert r["metrics"]["auc_pr"] > 0.3, f"{name} AUC-PR should exceed 0.3"
print("\n[ok] Checkpoint 1 passed — all three libraries trained\n")



## TASK 4 — VISUALISE the comparison


In [ ]:
names = list(results.keys())
auc_pr_values = [results[n]["metrics"]["auc_pr"] for n in names]
auc_roc_values = [results[n]["metrics"]["auc_roc"] for n in names]
log_loss_values = [results[n]["metrics"]["log_loss"] for n in names]
time_values = [results[n]["train_time"] for n in names]

fig = go.Figure()
# TODO: Add two grouped Bar traces: one for AUC-PR on yaxis y1, one for
# train time on yaxis y2.
____
____
fig.update_layout(
    title="Boosting Library Comparison — Singapore Credit Default",
    barmode="group",
    yaxis=dict(title="AUC-PR", range=[0, max(auc_pr_values) * 1.25]),
    yaxis2=dict(title="Train time (s)", overlaying="y", side="right"),
)
viz_path = OUTPUT_DIR / "ex4_03_library_comparison.html"
fig.write_html(viz_path)
print(f"  Saved: {viz_path}")

print("\n  --- Library Comparison Table ---")
print(
    f"  {'Library':<10} {'AUC-ROC':>10} {'AUC-PR':>10} {'Log Loss':>10} {'Time (s)':>10}"
)
print("  " + "─" * 56)
for n, auc_pr, auc_roc, ll, t in zip(
    names, auc_pr_values, auc_roc_values, log_loss_values, time_values
):
    print(f"  {n:<10} {auc_roc:>10.4f} {auc_pr:>10.4f} {ll:>10.4f} {t:>10.2f}")



### Checkpoint 2


In [ ]:
best_name = max(results.items(), key=lambda kv: kv[1]["metrics"]["auc_pr"])[0]
fastest_name = min(results.items(), key=lambda kv: kv[1]["train_time"])[0]
assert best_name in results
assert fastest_name in results
print("\n[ok] Checkpoint 2 passed — comparison computed\n")



## TASK 5 — APPLY: FairPrice Fraud Detection Library Choice


In [ ]:
# SCENARIO: NTUC FairPrice, 160 stores, ~4M transactions/day, 40%
# categorical features including 12,000-level SKU bundles and 9,000-
# level card BINs, 0.2% fraud rate, nightly re-train on 120M rows.
#
# LIBRARY CHOICE: CatBoost — ordered boosting handles high-cardinality
# categoricals without manual target encoding. LightGBM/XGBoost would
# require a cross-validated target encoding pipeline that owners own
# forever. 3-5x train-time penalty is acceptable in a nightly window.
#
# BUSINESS IMPACT: ~S$4.5B card volume/year → S$1.3-2.7M/year in fraud
# losses. AUC-PR 0.65 recovers ~40% → S$800K-1.1M/year avoided losses
# against S$80K/year infrastructure cost. 10-14x ROI.



## REFLECTION


[x] Trained XGBoost, LightGBM, CatBoost with matched hyperparameters
  [x] Read each library's design (exact / histogram / ordered boosting)
  [x] Compared AUC-PR and train time side-by-side
  [x] Best by AUC-PR: {best_name} | Fastest: {fastest_name}
  [x] Matched library choice to data shape using FairPrice scenario

  Next: 04_boosting_tuning.py — sweeps, heatmaps, and early stopping.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

